In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
BASE_DIR   = '/content/drive/MyDrive/vcc26_split622'
TRAIN_DIR  = BASE_DIR + '/train'
VAL_DIR    = BASE_DIR + '/validation'
TEST_DIR   = BASE_DIR + '/test'
SAVE_DIR   = BASE_DIR + '/h5'

os.makedirs(SAVE_DIR, exist_ok=True)

IMG_HEIGHT  = 224
IMG_WIDTH   = 224
BATCH_SIZE  = 32

# Phase 1 - Feature Extraction
EPOCHS_FE   = 10
LR_FE       = 1e-3

# Phase 2 - Fine Tuning
EPOCHS_FT   = 20
LR_FT       = 1e-5

In [ ]:
class_names = sorted([
    c for c in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, c))
])

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)
print("")

for i, name in enumerate(class_names):
    count = len(os.listdir(os.path.join(TRAIN_DIR, name)))
    print(f"  Class {i:02d} | {name} | {count} images")

# Save labels.txt - you will need this file in the competition app
labels_path = BASE_DIR + '/labels.txt'
with open(labels_path, 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print("")
print("labels.txt saved to:", labels_path)

Number of classes: 25

  Class 00 | Be Prepared to Stop | 84 images
  Class 01 | Do Not Enter | 77 images
  Class 02 | Left Curve Ahead | 101 images
  Class 03 | Left Turn Only | 76 images
  Class 04 | Left Turn Yield on Green | 87 images
  Class 05 | No Left Turn | 82 images
  Class 06 | No Parking | 85 images
  Class 07 | No Parking Double Arrow | 97 images
  Class 08 | No Parking Left Arrow | 96 images
  Class 09 | No Right Turn | 94 images
  Class 10 | No Turn On Red | 76 images
  Class 11 | No U Turn | 96 images
  Class 12 | None | 73 images
  Class 13 | One Way Left | 86 images
  Class 14 | One Way Right | 88 images
  Class 15 | Pedestrian Crossing | 84 images
  Class 16 | Right Curve Ahead | 92 images
  Class 17 | Right Turn Only | 79 images
  Class 18 | School Crossing | 85 images
  Class 19 | Speed Limit | 81 images
  Class 20 | Stop | 74 images
  Class 21 | Straight or Left Turn Only | 84 images
  Class 22 | Straight or Right Turn Only | 95 images
  Class 23 | When Flashing |

In [ ]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels      = 'inferred',
    label_mode  = 'categorical',
    class_names = class_names,
    image_size  = (IMG_HEIGHT, IMG_WIDTH),
    batch_size  = BATCH_SIZE,
    shuffle     = True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels      = 'inferred',
    label_mode  = 'categorical',
    class_names = class_names,
    image_size  = (IMG_HEIGHT, IMG_WIDTH),
    batch_size  = BATCH_SIZE,
    shuffle     = False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels      = 'inferred',
    label_mode  = 'categorical',
    class_names = class_names,
    image_size  = (IMG_HEIGHT, IMG_WIDTH),
    batch_size  = BATCH_SIZE,
    shuffle     = False
)

print("Train batches    :", len(train_ds))
print("Validation batches:", len(val_ds))
print("Test batches     :", len(test_ds))

Found 2152 files belonging to 25 classes.
Found 709 files belonging to 25 classes.
Found 741 files belonging to 25 classes.
Train batches    : 68
Validation batches: 23
Test batches     : 24


In [ ]:
# Augmentation layer applied during training only
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.3),    # handles changing light conditions
    tf.keras.layers.RandomContrast(0.2),
], name='augmentation')

# Rescale to [0, 1]
rescale = tf.keras.layers.Rescaling(1.0 / 255)

def prepare_train(images, labels):
    images = augmentation(images, training=True)
    images = rescale(images)
    return images, labels

def prepare_eval(images, labels):
    images = rescale(images)
    return images, labels

# AUTOTUNE prefetches batches so GPU never sits idle waiting for data
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(prepare_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(prepare_eval,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(prepare_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print("Datasets ready with augmentation and prefetching")

Datasets ready with augmentation and prefetching


In [ ]:
# Load VGG16 without the top classification layers
base_model = VGG16(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
)

# Phase 1 - freeze all base model layers
base_model.trainable = False

# Build the top classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 14,882,137 (56.77 MB)

 Trainable params: 167,449 (654.10 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
model.compile(
    optimizer = Adam(learning_rate=LR_FE),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

callbacks_fe = [
    EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = 5,
        restore_best_weights = True
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 3,
        verbose  = 1
    ),
    ModelCheckpoint(
        filepath       = SAVE_DIR + '/vgg16_FE_best.keras',
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 1
    )
]

print("Starting Phase 1 - Feature Extraction")

history_fe = model.fit(
    train_ds,
    epochs          = EPOCHS_FE,
    validation_data = val_ds,
    callbacks       = callbacks_fe
)

print("Phase 1 done")
print("Best val accuracy:", max(history_fe.history['val_accuracy']))

Starting Phase 1 - Feature Extraction
Epoch 1/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.0421 - loss: 3.2781
Epoch 1: val_accuracy improved from None to 0.15092, saving model to /content/drive/MyDrive/vcc26_split622/h5/vgg16_FE_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/vcc26_split622/h5/vgg16_FE_best.keras
68/68 ━━━━━━━━━━━━━━━━━━━━ 56s 593ms/step - accuracy: 0.0702 - loss: 3.2083 - val_accuracy: 0.1509 - val_loss: 3.0255 - learning_rate: 0.0010
Epoch 2/10
67/68 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.1212 - loss: 2.9860
Epoch 2: val_accuracy improved from 0.15092 to 0.31171, saving model to /content/drive/MyDrive/vcc26_split622/h5/vgg16_FE_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/vcc26_split622/h5/vgg16_FE_best.keras
68/68 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.1408 - loss: 2.8946 - val_accuracy: 0.3117 - val_loss: 2.4837 - learning_rate: 0.0010
Epoch 3/10
67/68 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - 

In [ ]:
for layer in base_model.layers[-4:]:
    layer.trainable = True

model.compile(
    optimizer = Adam(learning_rate=LR_FT),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

callbacks_ft = [
    EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = 7,
        restore_best_weights = True
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 3,
        verbose  = 1
    ),
    ModelCheckpoint(
        filepath       = SAVE_DIR + '/vcc26_vgg16_FE2FineT.keras',
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 1
    )
]

print("Starting Phase 2 - Fine Tuning")

history_ft = model.fit(
    train_ds,
    epochs          = EPOCHS_FT,
    validation_data = val_ds,
    callbacks       = callbacks_ft
)

print("Phase 2 done")
print("Best val accuracy:", max(history_ft.history['val_accuracy']))

Starting Phase 2 - Fine Tuning
Epoch 1/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.6317 - loss: 0.9218
Epoch 1: val_accuracy improved from None to 0.77574, saving model to /content/drive/MyDrive/vcc26_split622/h5/vcc26_vgg16_FE2FineT.keras

Epoch 1: finished saving model to /content/drive/MyDrive/vcc26_split622/h5/vcc26_vgg16_FE2FineT.keras
68/68 ━━━━━━━━━━━━━━━━━━━━ 23s 211ms/step - accuracy: 0.6659 - loss: 0.8399 - val_accuracy: 0.7757 - val_loss: 0.5031 - learning_rate: 1.0000e-05
Epoch 2/20
67/68 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.6851 - loss: 0.7156
Epoch 2: val_accuracy improved from 0.77574 to 0.82511, saving model to /content/drive/MyDrive/vcc26_split622/h5/vcc26_vgg16_FE2FineT.keras

Epoch 2: finished saving model to /content/drive/MyDrive/vcc26_split622/h5/vcc26_vgg16_FE2FineT.keras
68/68 ━━━━━━━━━━━━━━━━━━━━ 8s 113ms/step - accuracy: 0.6849 - loss: 0.7074 - val_accuracy: 0.8251 - val_loss: 0.4610 - learning_rate: 1.0000e-05
Epoch 3/20
67/68 ━━━━━━━

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print("Test Loss     :", round(test_loss, 4))
print("Test Accuracy :", round(test_accuracy * 100, 2), "%")

24/24 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 0.9798 - loss: 0.1260
Test Loss     : 0.126
Test Accuracy : 97.98 %


In [ ]:
final_model_path = SAVE_DIR + '/vcc26_vgg16_FE2FineT.keras'
model.save(final_model_path)
print("Final model saved to:", final_model_path)

Final model saved to: /content/drive/MyDrive/vcc26_split622/h5/vcc26_vgg16_FE2FineT.keras
